# 청킹 전략 비교: RecursiveCharacterTextSplitter vs MarkdownHeaderTextSplitter

## 가설
마크다운 헤더 단위로 자르면 Q&A 한 덩어리가 유지되어 검색 품질이 좋아진다.  
길이 기반(Recursive)은 섹션 중간이 잘려 맥락이 끊기는 케이스가 발생할 것.

## 비교 쿼리 3개
| # | 쿼리 | 예상 차이 |
|---|------|-----------|
| Q1 | 양조통에 넣을 수 있는 재료와 산출 제품 목록 | 재료-산출물 대응표가 한 섹션에 → 헤더 기반이 표 전체 보존 가능성 |
| Q2 | 봄 작물 종류와 각각의 성장 기간 | 다수 작물 목록 → 길이 기반은 목록 중간 절단 가능 |
| Q3 | 낚시 미끼 종류와 각각의 효과 | 미끼 정보가 섹션별 → 헤더 기반이 섹션 단위 보존 |

> **주의**: 실제 차이는 수집된 위키 문서 구조에 따라 달라질 수 있음.

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

from pathlib import Path
DOCS_DIR = Path('data/docs_sample')
print(f'문서 디렉토리: {DOCS_DIR}')
print(f'마크다운 파일 수: {len(list(DOCS_DIR.glob("*.md")))}')

문서 디렉토리: data/docs_sample
마크다운 파일 수: 24


## 전략 A: RecursiveCharacterTextSplitter

In [2]:
from rag.wiki import WikiRetrievalChain

wiki_recursive = WikiRetrievalChain(
    source_uri=str(DOCS_DIR),
    splitter_type='recursive',
    chunk_size=800,
    chunk_overlap=150,
).create_chain()

retriever_recursive = wiki_recursive.retriever

Loaded: crafting_overview.md (제작)
Loaded: farming_ancient_seed.md (농사)
Loaded: farming_crops.md (농사)
Loaded: farming_crops_fall.md (농사)
Loaded: farming_crops_spring.md (농사)
Loaded: farming_crops_summer.md (농사)
Loaded: farming_crops_winter.md (농사)
Loaded: farming_greenhouse.md (농사)
Loaded: farming_keg.md (농사)
Loaded: farming_lightning_rod.md (농사)
Loaded: farming_preserves_jar.md (농사)
Loaded: fishing_fish_list.md (낚시)
Loaded: fishing_overview.md (낚시)
Loaded: fishing_tackle.md (낚시)
Loaded: foraging_overview.md (채집)
Loaded: livestock_animals.md (축산)
Loaded: livestock_barn.md (축산)
Loaded: livestock_coop.md (축산)
Loaded: livestock_mayonnaise.md (축산)
Loaded: mining_dwarf_scroll1.md (광산)
Loaded: mining_dwarf_scroll2.md (광산)
Loaded: mining_overview.md (광산)
Loaded: villagers_hearts.md (주민)
Loaded: villagers_marriage.md (주민)

Loaded 24 documents, 0 failed.
Split into 1655 chunks (recursive, size=800)
FAISS index saved to cache


## 전략 B: MarkdownHeaderTextSplitter

In [3]:
wiki_markdown = WikiRetrievalChain(
    source_uri=str(DOCS_DIR),
    splitter_type='markdown_header',
).create_chain()

retriever_markdown = wiki_markdown.retriever

Loaded: crafting_overview.md (제작)
Loaded: farming_ancient_seed.md (농사)
Loaded: farming_crops.md (농사)
Loaded: farming_crops_fall.md (농사)
Loaded: farming_crops_spring.md (농사)
Loaded: farming_crops_summer.md (농사)
Loaded: farming_crops_winter.md (농사)
Loaded: farming_greenhouse.md (농사)
Loaded: farming_keg.md (농사)
Loaded: farming_lightning_rod.md (농사)
Loaded: farming_preserves_jar.md (농사)
Loaded: fishing_fish_list.md (낚시)
Loaded: fishing_overview.md (낚시)
Loaded: fishing_tackle.md (낚시)
Loaded: foraging_overview.md (채집)
Loaded: livestock_animals.md (축산)
Loaded: livestock_barn.md (축산)
Loaded: livestock_coop.md (축산)
Loaded: livestock_mayonnaise.md (축산)
Loaded: mining_dwarf_scroll1.md (광산)
Loaded: mining_dwarf_scroll2.md (광산)
Loaded: mining_overview.md (광산)
Loaded: villagers_hearts.md (주민)
Loaded: villagers_marriage.md (주민)

Loaded 24 documents, 0 failed.
Split into 339 chunks (markdown_header)
FAISS index saved to cache


## 비교 함수 정의

In [4]:
def compare_retrievers(query: str, k: int = 3):
    """두 retriever의 검색 결과를 나란히 출력."""
    print(f'\n{'='*60}')
    print(f'Query: {query}')
    print('='*60)

    docs_r = retriever_recursive.invoke(query)
    docs_m = retriever_markdown.invoke(query)

    print(f'\n[A] RecursiveCharacterTextSplitter (상위 {len(docs_r)}개)')
    for i, doc in enumerate(docs_r, 1):
        src = doc.metadata.get('source', '?')
        cat = doc.metadata.get('category', '')
        content_preview = doc.page_content[:200].replace('\n', ' ')
        print(f'  #{i} [{cat}] {Path(src).name}')
        print(f'      청크 길이: {len(doc.page_content)}자')
        print(f'      내용 앞부분: {content_preview}...')

    print(f'\n[B] MarkdownHeaderTextSplitter (상위 {len(docs_m)}개)')
    for i, doc in enumerate(docs_m, 1):
        src = doc.metadata.get('source', '?')
        cat = doc.metadata.get('category', '')
        h1 = doc.metadata.get('Header 1', '')
        h2 = doc.metadata.get('Header 2', '')
        section = f'{h1} > {h2}' if h2 else h1
        content_preview = doc.page_content[:200].replace('\n', ' ')
        print(f'  #{i} [{cat}] {Path(src).name} § {section}')
        print(f'      청크 길이: {len(doc.page_content)}자')
        print(f'      내용 앞부분: {content_preview}...')

## 비교 쿼리 Q1: 양조통 재료와 산출 제품

In [5]:
compare_retrievers('양조통에 넣을 수 있는 재료와 각각 만들어지는 제품은?')


Query: 양조통에 넣을 수 있는 재료와 각각 만들어지는 제품은?

[A] RecursiveCharacterTextSplitter (상위 6개)
  #1 [농사] farming_crops.md
      청크 길이: 696자
      내용 앞부분: ## 장인 제작품  판매 가치를 올리기 위해 작물을 [장인 제작품](/%EC%9E%A5%EC%9D%B8_%EC%A0%9C%EC%9E%91%ED%92%88 "장인 제작품")으로 만들 수 있습니다. 대부분의 [과일](/%EA%B3%BC%EC%9D%BC "과일")과 [야채](/%EC%95%BC%EC%B1%84 "야채")는 [절임통](/%EC%A0%88%EC%9E...
  #2 [농사] farming_preserves_jar.md
      청크 길이: 555자
      내용 앞부분: **절임통**은 [장인 제작품](/%EC%9E%A5%EC%9D%B8_%EC%A0%9C%EC%9E%91%ED%92%88 "장인 제작품")을 만들 수 있는 [장인 설비](/%EC%A0%9C%EC%9E%91#.EC.9E.A5.EC.9D.B8_.EC.84.A4.EB.B9.84 "제작")입니다. [과일](/%EA%B3%BC%EC%9D%BC "과일")을 [젤리](/%...
  #3 [농사] farming_keg.md
      청크 길이: 728자
      내용 앞부분: **술통**은 [장인 제작품](/%EC%9E%A5%EC%9D%B8_%EC%A0%9C%EC%9E%91%ED%92%88 "장인 제작품")을 만들어내는 [장인 설비](/%EC%A0%9C%EC%9E%91#.EC.9E.A5.EC.9D.B8_.EC.84.A4.EB.B9.84 "제작")입니다. 술통은 마을 회관 [식료품 저장실](/%EA%BE%B8%EB%9F%AC%EB...
  #4 [축산] livestock_mayonnaise.md
      청크 길이: 671자
      내용 앞부분: | [장인 제작품](/%EC%9E%A5%EC%9D%B8_%EC%A0%9C%EC%9E%91%ED%92%

## 비교 쿼리 Q2: 봄 작물 종류와 성장 기간

In [6]:
compare_retrievers('봄에 심을 수 있는 작물의 종류와 각각의 성장 기간은?')


Query: 봄에 심을 수 있는 작물의 종류와 각각의 성장 기간은?

[A] RecursiveCharacterTextSplitter (상위 6개)
  #1 [농사] farming_crops.md
      청크 길이: 7자
      내용 앞부분: ## 봄 작물...
  #2 [농사] farming_crops.md
      청크 길이: 8자
      내용 앞부분: ## 겨울 작물...
  #3 [채집] foraging_overview.md
      청크 길이: 5자
      내용 앞부분: ### 봄...
  #4 [농사] farming_crops.md
      청크 길이: 126자
      내용 앞부분: :   \* [고대 씨앗 (유물)](/%EA%B3%A0%EB%8C%80_%EC%94%A8%EC%95%97_(%EC%9C%A0%EB%AC%BC) "고대 씨앗 (유물)")로 만들 수 있고 봄, 여름, 가을 동안 심을 수 있습니다....
  #5 [농사] farming_crops.md
      청크 길이: 39자
      내용 앞부분: :   1 봄 1일에 심었을 때 :   2 봄 13/14일에 심었을 때...
  #6 [농사] farming_crops_winter.md
      청크 길이: 755자
      내용 앞부분: ## 작물  [겨울 씨앗 모음](/%EA%B2%A8%EC%9A%B8_%EC%94%A8%EC%95%97_%EB%AA%A8%EC%9D%8C "겨울 씨앗 모음")을 제외하고, 겨울에 심을 수 있는 [작물](/%EC%9E%91%EB%AC%BC "작물")이 없습니다.  | 이미지 | 이름 | 설명 | 재료 | 레시피 | | --- | --- | --- | --- |...

[B] MarkdownHeaderTextSplitter (상위 6개)
  #1 [농사] farming_crops_winter.md § Farming Crops Winter > 작물
      청크 길이: 753자
      내용 앞부분: ## 작물   [겨울 씨앗 모

## 비교 쿼리 Q3: 낚시 미끼 종류와 효과

In [7]:
compare_retrievers('낚시 미끼의 종류와 각각의 효과는?')


Query: 낚시 미끼의 종류와 각각의 효과는?

[A] RecursiveCharacterTextSplitter (상위 6개)
  #1 [낚시] fishing_tackle.md
      청크 길이: 442자
      내용 앞부분: | [낚시](/%EB%82%9A%EC%8B%9C "낚시") | 미끼 | [도전 미끼](/%EB%8F%84%EC%A0%84_%EB%AF%B8%EB%81%BC "도전 미끼") • [디럭스 미끼](/%EB%94%94%EB%9F%AD%EC%8A%A4_%EB%AF%B8%EB%81%BC "디럭스 미끼") • [마법 미끼](/%EB%A7%88%EB%B2%95_%EB%A...
  #2 [농사] farming_keg.md
      청크 길이: 470자
      내용 앞부분: | [낚시](/%EB%82%9A%EC%8B%9C "낚시") | [미끼](/%EB%AF%B8%EB%81%BC "미끼") | [도전 미끼](/%EB%8F%84%EC%A0%84_%EB%AF%B8%EB%81%BC "도전 미끼") • [디럭스 미끼](/%EB%94%94%EB%9F%AD%EC%8A%A4_%EB%AF%B8%EB%81%BC "디럭스 미끼") • [마법 미...
  #3 [농사] farming_lightning_rod.md
      청크 길이: 470자
      내용 앞부분: | [낚시](/%EB%82%9A%EC%8B%9C "낚시") | [미끼](/%EB%AF%B8%EB%81%BC "미끼") | [도전 미끼](/%EB%8F%84%EC%A0%84_%EB%AF%B8%EB%81%BC "도전 미끼") • [디럭스 미끼](/%EB%94%94%EB%9F%AD%EC%8A%A4_%EB%AF%B8%EB%81%BC "디럭스 미끼") • [마법 미...
  #4 [농사] farming_preserves_jar.md
      청크 길이: 470자
      내용 앞부분: | [낚시](/%EB%82%9A%EC%8B%9C "낚시") | [미끼](/%EB%AF%B8%EB%81%BC "미끼")

## 청크 통계 비교

In [8]:
from rag.wiki import WikiRetrievalChain
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

loader_r = WikiRetrievalChain(source_uri=str(DOCS_DIR), splitter_type='recursive', chunk_size=800, chunk_overlap=150)
docs_raw = loader_r.load_documents(str(DOCS_DIR))

# Recursive 통계
splitter_r = loader_r.create_text_splitter()
chunks_r = loader_r.split_documents(docs_raw, splitter_r)
sizes_r = [len(c.page_content) for c in chunks_r]

# MarkdownHeader 통계
loader_m = WikiRetrievalChain(source_uri=str(DOCS_DIR), splitter_type='markdown_header')
splitter_m = loader_m.create_text_splitter()
chunks_m = loader_m.split_documents(docs_raw, splitter_m)
sizes_m = [len(c.page_content) for c in chunks_m]

import statistics
print('=== 청크 통계 ===')
print(f'[A] Recursive  : {len(chunks_r)}개, 평균 {statistics.mean(sizes_r):.0f}자, 중앙값 {statistics.median(sizes_r):.0f}자')
print(f'[B] MarkdownHeader: {len(chunks_m)}개, 평균 {statistics.mean(sizes_m):.0f}자, 중앙값 {statistics.median(sizes_m):.0f}자')

Loaded: crafting_overview.md (제작)
Loaded: farming_ancient_seed.md (농사)
Loaded: farming_crops.md (농사)
Loaded: farming_crops_fall.md (농사)
Loaded: farming_crops_spring.md (농사)
Loaded: farming_crops_summer.md (농사)
Loaded: farming_crops_winter.md (농사)
Loaded: farming_greenhouse.md (농사)
Loaded: farming_keg.md (농사)
Loaded: farming_lightning_rod.md (농사)
Loaded: farming_preserves_jar.md (농사)
Loaded: fishing_fish_list.md (낚시)
Loaded: fishing_overview.md (낚시)
Loaded: fishing_tackle.md (낚시)
Loaded: foraging_overview.md (채집)
Loaded: livestock_animals.md (축산)
Loaded: livestock_barn.md (축산)
Loaded: livestock_coop.md (축산)
Loaded: livestock_mayonnaise.md (축산)
Loaded: mining_dwarf_scroll1.md (광산)
Loaded: mining_dwarf_scroll2.md (광산)
Loaded: mining_overview.md (광산)
Loaded: villagers_hearts.md (주민)
Loaded: villagers_marriage.md (주민)

Loaded 24 documents, 0 failed.
Split into 1655 chunks (recursive, size=800)
Split into 339 chunks (markdown_header)
=== 청크 통계 ===
[A] Recursive  : 1655개, 평균 503자, 중앙값 541자
[B

## 비교 결과 정리

| 기준 | RecursiveCharacterTextSplitter | MarkdownHeaderTextSplitter |
|------|-------------------------------|---------------------------|
| 청크 수 | 1,655개 | 339개 |
| 평균 크기 | 503자 | 2,303자 |
| 중앙값 크기 | 541자 | 1,275자 |
| Q1 (양조통 재료·산출물) | farming_keg.md 728자 청크 검색 — 기본 정보 포함 | farming_keg.md 전체 섹션 1,645자 보존 — 재료·산출물 대응표 온전히 유지 |
| Q2 (봄 작물 성장 기간) | 상위 결과가 7자("## 봄 작물"), 5자("### 봄") 등 헤더만 잘린 무의미 청크 — 실질 정보 없음 | farming_crops_spring.md 봄 시즌 전체 컨텍스트 정상 검색 |
| Q3 (낚시 미끼 종류·효과) | farming_keg.md 등 관련 없는 문서에서 "미끼" 언급 부분이 상위 노출 — 노이즈 다수 | fishing_tackle.md § 미끼 아이템 2,099자 — 효과 상세표 온전히 검색 |
| 선택 전략 | - | **MarkdownHeaderTextSplitter 채택** |

### 선택 이유

Q2에서 Recursive가 7자·5자짜리 헤더만 있는 빈 청크를 상위로 올린 것이 결정적이었다.
`chunk_size=800`으로 자를 때 헤더 바로 뒤 내용이 다음 청크로 넘어가면서 헤더만 남은 조각이 생성된 것으로, 검색 결과로 쓸 수 없는 수준이었다.

반면 MarkdownHeaderTextSplitter는 문서 구조를 보존하므로:
- Q1: 술통 재료-산출물 대응표가 하나의 청크(1,645자)에 온전히 담겨 있었다.
- Q3: 미끼 효과 상세표(2,099자) 전체가 단일 청크로 검색됐다.

단점으로 평균 청크 크기가 2,303자로 크기 때문에 LLM 입력 토큰이 늘어나고, 최대 3,641자짜리 청크도 존재한다.
그러나 스타듀밸리 위키처럼 헤더 단위로 정보가 명확히 구분된 문서에서는 구조 보존의 이점이 토큰 비용을 상쇄한다고 판단했다.